# Patrón de Comportamiento: Template Method

## Introducción
El patrón Template Method define el esqueleto de un algoritmo en una operación, dejando algunos pasos a las subclases.

## Objetivos
- Comprender cómo definir algoritmos con pasos personalizables.
- Identificar cuándo es útil el patrón Template Method.
- Comparar la solución con y sin el patrón.

## Ejemplo de la vida real
**Contexto: Proceso de compra online**
El proceso de compra puede tener pasos comunes (selección, pago, envío), pero cada tienda puede personalizar algunos pasos.

**¿Dónde se usa en proyectos reales?**
En frameworks, procesamiento de datos, sistemas de workflow, etc.

## Sin patrón Template Method (forma errónea)
El algoritmo está completamente definido en una sola clase, dificultando la personalización.

In [1]:
class CompraOnline:
    def __init__(self, metodo_pago: str, destino: str) -> None:
        self.metodo_pago = metodo_pago
        self.destino = destino

    def comprar(self) -> None:
        self.seleccionar()
        self.pagar()
        self.enviar()

    def seleccionar(self) -> None:
        print("Seleccionando producto")

    def pagar(self) -> None:
        if self.metodo_pago == "datafono":
            print("Pagando con datáfono")
        elif self.metodo_pago == "tarjeta":
            print("Pagando con tarjeta")
        elif self.metodo_pago == "pse":
            print("Pagando con PSE")
        else:
            print("Método de pago no válido")

    def enviar(self) -> None:
        if self.destino == "tienda":
            print("Enviando producto a la tienda")
        else:
            print("En casa")

## Con patrón Template Method (forma correcta)
Algunos pasos se dejan a las subclases para personalización.

In [2]:
import abc


class CompraOnlineGood(abc.ABC):
    def comprar(self) -> None:
        self.seleccionar()
        self.pagar()
        self.enviar()

    def seleccionar(self) -> None:
        print("Seleccionando producto")

    @abc.abstractmethod
    def pagar(self) -> None:
        pass

    @abc.abstractmethod
    def enviar(self) -> None:
        pass


class PagoPseEnTienda(CompraOnlineGood):
    def pagar(self) -> None:
        print("Pagando con PSE")

    def enviar(self) -> None:
        print("Enviando a la tienda")


class PagoTarjetaCasa(CompraOnlineGood):
    def pagar(self) -> None:
        print("Pagando con tarjeta")

    def enviar(self) -> None:
        print("Enviando a la casa")


compra_a = PagoPseEnTienda()
compra_a.comprar()

compra_b = PagoTarjetaCasa()
compra_b.comprar()

Seleccionando producto
Pagando con PSE
Enviando a la tienda
Seleccionando producto
Pagando con tarjeta
Enviando a la casa


## UML del patrón Template Method
```plantuml
@startuml
class CompraOnline {
    + comprar()
    + seleccionar()
    + pagar()
    + enviar()
}
CompraOnline <|-- CompraTiendaA
CompraOnline <|-- CompraTiendaB
@enduml
```

## Otro ejemplo de la vida real: Pipeline ETL (Extraer, Transformar, Cargar)
**Contexto:** en ingeniería de datos, un pipeline ETL siempre sigue el mismo esqueleto (extraer → transformar → cargar), pero la forma de **extraer** cambia según la fuente (un CSV, una API REST, una base de datos) y a veces la **transformación** también necesita un ajuste específico por fuente. El orden de los pasos nunca cambia; lo que cambia es cómo se ejecuta cada paso.

### Sin patrón (forma errónea)
Un `if/elif` por tipo de fuente mezcla toda la lógica de extracción dentro de un único método.

In [3]:
class PipelineETL:
    def ejecutar(self, fuente: str) -> None:
        if fuente == 'csv':
            datos = 'datos leídos de archivo.csv'
        elif fuente == 'api':
            datos = 'datos leídos de API REST'
        else:
            raise ValueError('Fuente no soportada')
        transformados = datos.upper()
        print(f'Cargando a bodega de datos: {transformados}')

# Agregar una fuente nueva (ej. base de datos) obliga a editar este método existente
pipeline = PipelineETL()
pipeline.ejecutar('csv')
pipeline.ejecutar('api')

Cargando a bodega de datos: DATOS LEÍDOS DE ARCHIVO.CSV
Cargando a bodega de datos: DATOS LEÍDOS DE API REST


### Con patrón (forma correcta)
`ejecutar()` define el esqueleto fijo (extraer → transformar → cargar). `extraer()` es obligatorio para cada subclase; `transformar()` tiene una implementación por defecto que una subclase puede sobrescribir solo si lo necesita.

In [4]:
import abc

class PipelineETL(abc.ABC):
    def ejecutar(self) -> None:
        datos = self.extraer()
        transformados = self.transformar(datos)
        self.cargar(transformados)

    @abc.abstractmethod
    def extraer(self) -> str:
        ...

    def transformar(self, datos: str) -> str:
        return datos.upper()

    def cargar(self, datos: str) -> None:
        print(f'Cargando a bodega de datos: {datos}')


class PipelineCSV(PipelineETL):
    def extraer(self) -> str:
        return 'datos leídos de archivo.csv'


class PipelineAPI(PipelineETL):
    def extraer(self) -> str:
        return 'datos leídos de API REST'

    def transformar(self, datos: str) -> str:
        return datos.upper() + ' (normalizado desde JSON)'


PipelineCSV().ejecutar()
PipelineAPI().ejecutar()

Cargando a bodega de datos: DATOS LEÍDOS DE ARCHIVO.CSV
Cargando a bodega de datos: DATOS LEÍDOS DE API REST (normalizado desde JSON)


### UML del ejemplo de pipeline ETL
```plantuml
@startuml
abstract class PipelineETL {
    + ejecutar()
    + extraer()
    + transformar(datos)
    + cargar(datos)
}
class PipelineCSV
class PipelineAPI
PipelineETL <|-- PipelineCSV
PipelineETL <|-- PipelineAPI
@enduml
```

### ¿Dónde más se usa Template Method?
- **Pipelines de datos (ETL/ELT):** exactamente este ejemplo — Airflow, dbt y frameworks similares estructuran tareas con un esqueleto fijo y pasos personalizables por fuente.
- **Frameworks de testing:** `setUp()` → `test_*()` → `tearDown()` es un Template Method: el framework controla el orden, tú solo implementas los pasos.
- **Procesos de compra online:** el ejemplo con el que abre este notebook — seleccionar, pagar, enviar, donde el método de pago y de envío varían por tienda.
- **Algoritmos de ordenamiento con "hooks":** muchas librerías de sorting fijan el algoritmo general pero permiten personalizar el criterio de comparación.
- **Ciclo de vida de componentes en frameworks web:** frameworks como Django (`dispatch()` en las vistas) o Spring definen el orden de las fases y delegan detalles específicos a las subclases/implementaciones del desarrollador.

**Ejercicio de reflexión:** ¿por qué `transformar()` no es un método abstracto (`@abc.abstractmethod`) como `extraer()`? ¿Qué diferencia práctica hay entre un paso obligatorio y un paso opcional con implementación por defecto (a veces llamado "hook") en Template Method?

## Actividad
Crea una plantilla para el proceso de registro de usuarios, permitiendo personalizar el paso de verificación.

---
## Explicación de conceptos clave
- **Algoritmo flexible:** Permite personalizar pasos específicos.
- **Reutilización:** El esqueleto del algoritmo se reutiliza.
- **Aplicación en la vida real:** Útil en frameworks, procesamiento de datos y sistemas de workflow.

## Conclusión
El patrón Template Method es ideal para definir algoritmos con pasos personalizables y reutilizables.